# Simulação de Cavidade Elíptica para NV-Maser

Este notebook tem como objetivo modelar e simular as propriedades eletromagnéticas de uma cavidade ressonante elíptica preenchida com dielétrico (como safira), focada em aplicações de NV-Maser. 

O objetivo principal é encontrar as condições físicas (dimensões e permissividade) para que ocorra a **degenerescência** entre os modos $TE_{111}$ e $TM_{010}$ exatamente na frequência alvo de transição do centro NV do diamante ($2.87$ GHz).

### Passo 1: Importação de Bibliotecas e Constantes
Iniciamos importando as ferramentas matemáticas e gráficas necessárias, além de definir as constantes físicas e raízes das funções de Bessel.

* $c_0 = 3 \times 10^{11}$ mm/s (Velocidade da luz no vácuo em mm/s)
* $X_{01} = 2.40483$ (Primeira raiz de $J_0(x)$, dita o corte TM)
* $X'_{11} = 1.84118$ (Primeira raiz da derivada de $J_1(x)$, dita o corte TE)
* $f_{alvo} = 2.87$ GHz (Frequência do centro NV)

In [19]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy.special import jv, jvp
import threading, time
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── constants ─────────────────────────────────────────────────
C0 = 3e11             # Velocidade da luz (mm/s)
X01 = 2.40483         # Raiz para TM010
XP11 = 1.84118        # Raiz para TE111
TARGET = 2.87         # Frequência alvo em GHz

### Passo 2: Geometria e Frequências de Ressonância

Para uma cavidade elíptica com semi-eixo maior $a$ e excentricidade $e$, definimos um **raio efetivo** $R_{eff}$ para aproximar o comportamento cilíndrico:
$$R_{eff} = a(1 - e^2)^{1/4}$$

A frequência do modo **TM$_{010}$** independe do comprimento da cavidade ($L$) e é dada por:
$$f_{TM_{010}} = \frac{c_0 X_{01}}{2\pi R_{eff} \sqrt{\varepsilon_r}}$$

Para o modo **TE$_{111}$**, a frequência depende do comprimento $L$. Calculamos primeiro a frequência de corte $f_c$ e o termo longitudinal $f_z$:
$$f_{c} = \frac{c_0 X'_{11}}{2\pi R_{eff} \sqrt{\varepsilon_r}}$$
$$f_z = \frac{c_0}{2L\sqrt{\varepsilon_r}}$$
A frequência de ressonância final é:
$$f_{TE_{111}} = \sqrt{f_c^2 + f_z^2}$$

In [20]:
# ── physics ───────────────────────────────────────────────────
def Reff(a, e):
    return a * np.power(1 - e**2, 0.25)

def fTM(a, e, er):
    return C0 * X01 / (2 * np.pi * Reff(a, e) * np.sqrt(er)) / 1e9

def fTEcutoff(a, e, er):
    return C0 * XP11 / (2 * np.pi * Reff(a, e) * np.sqrt(er)) / 1e9

def fTE(a, e, L, er):
    fc = fTEcutoff(a, e, er)
    fz = C0 / (2 * L * np.sqrt(er)) / 1e9
    return np.sqrt(fc**2 + fz**2)

### Passo 3: Busca pela Degenerescência e Volume Modal

Para que o Maser funcione otimamente, os modos $TE_{111}$ e $TM_{010}$ devem ser **degenerados** (terem a mesma frequência). Como o modo TM é fixo e o modo TE depende de $L$, usamos um método de **Bisseção** para varrer o espaço e encontrar o comprimento $L_{degen}$ exato onde as curvas se cruzam.

Também calculamos o volume do modo:
$$V = \pi a^2 \sqrt{1 - e^2} L$$

In [21]:
def Ldegen_bisect(a, e, er, Lo=20.0, Hi=500.0, tol=1e-6):
    """
    Busca binária (bisection) para encontrar o cruzamento TE111/TM010.
    Retorna um dicionário {L, f} ou None se não houver cruzamento no intervalo.
    """
    ftm = fTM(a, e, er)
    if fTE(a, e, Lo, er) <= ftm:
        return None   # TE já está abaixo do TM em Lo — sem cruzamento
    if fTE(a, e, Hi, er) >= ftm:
        return None   # TE nunca desce até o TM no limite Hi
        
    for _ in range(80):
        mid = (Lo + Hi) / 2
        if fTE(a, e, mid, er) > ftm:
            Lo = mid
        else:
            Hi = mid
        if Hi - Lo < tol:
            break
            
    Lc = (Lo + Hi) / 2
    return {'L': Lc, 'f': fTE(a, e, Lc, er)}

def mode_volume(a, e, L):
    return np.pi * a * a * np.sqrt(1 - e**2) * L   # mm³

### Passo 4: Distribuição dos Campos Eletromagnéticos (Funções de Bessel)

Os campos internos dependem das coordenadas espaciais ($r, \phi$). Eles são modelados usando funções de Bessel de primeiro tipo ($J_n$) e suas derivadas ($J'_n$). 

O campo magnético transversal $|H_\perp|$ do modo $TE_{11}$, vital para o acoplamento de spin no centro NV, depende das contribuições radiais e azimutais derivadas da equação de Helmholtz na geometria elíptica.

In [22]:
# ── fields ────────────────────────────────────────────────────
def _safe_r(X, Y):
    # Evita divisão por zero no centro da cavidade
    return np.where(X**2 + Y**2 < 1e-18, 1e-9, np.sqrt(X**2 + Y**2))

def ellipse_mask(X, Y, a, e):
    b = a * np.sqrt(1 - e**2)
    return X**2 / a**2 + Y**2 / b**2 <= 1.0

def F_Eperp(X, Y, a, e, phi0):
    kc = XP11 / Reff(a, e)
    r  = _safe_r(X, Y)
    ph = np.arctan2(Y, X) - phi0
    kr = kc * r
    Er   = jvp(1, kr, 1) * np.cos(ph)
    Ephi = jv(1, kr) / kr * np.sin(ph)
    return np.sqrt(Er**2 + Ephi**2)

def F_Hz(X, Y, a, e, phi0):
    kc = XP11 / Reff(a, e)
    r  = _safe_r(X, Y)
    return jv(1, kc * r) * np.cos(np.arctan2(Y, X) - phi0)

def F_Hperp(X, Y, a, e, phi0):
    kc  = XP11 / Reff(a, e)
    r   = _safe_r(X, Y)
    phi = np.arctan2(Y, X)
    dp  = phi - phi0
    kr  = kc * r
    dj  = jvp(1, kr, 1)
    j1r = jv(1, kr) / r
    gx  = kc * dj * np.cos(dp) * np.cos(phi) + j1r * np.sin(dp) * np.sin(phi)
    gy  = kc * dj * np.cos(dp) * np.sin(phi) - j1r * np.sin(dp) * np.cos(phi)
    return np.sqrt(gx**2 + gy**2)

def F_Ez(X, Y, a, e, _=None):
    return np.maximum(0, jv(0, X01 / Reff(a, e) * _safe_r(X, Y)))

def F_Hphi(X, Y, a, e, _=None):
    return np.maximum(0, jv(1, X01 / Reff(a, e) * _safe_r(X, Y)))

### Passo 5: Visualização de Dados (Gráficos e Parâmetro S11)

Nesta seção, definimos as funções que vão gerar as visualizações em Matplotlib.
* **Campos 2D**: Exibem a intensidade dos modos dentro da máscara elíptica.
* **Parâmetro $S_{11}$**: Simula a perda de retorno (reflexão) usando um modelo Lorentziano em que a cavidade ressoa produzindo um mergulho ("dip") na reflexão, mostrando a proximidade das frequências $TE$ e $TM$.
* **Varredura em L (L-Scan)**: Mostra visualmente onde ocorre o cruzamento de degenerescência.

In [23]:
# ── panel drawers ─────────────────────────────────────────────
def _overlay(ax, a, e):
    b   = a * np.sqrt(1 - e**2)
    foc = a * e
    th  = np.linspace(0, 2 * np.pi, 400)
    ax.plot(a * np.cos(th), b * np.sin(th), 'w-', lw=1, alpha=0.6)
    ax.plot([foc, -foc], [0, 0], 'o', color='#FF6B35', ms=5, zorder=5, label='foci')
    ax.plot(0, 0, '+', color='#7FBA00', ms=9, mew=2, zorder=5, label='centre')

def draw_field(ax, Ffn, a, e, phi0, title, cmap='inferno', signed=False, N=180):
    b = a * np.sqrt(1 - e**2)
    x = np.linspace(-a * 1.05, a * 1.05, N)
    y = np.linspace(-b * 1.05, b * 1.05, N)
    X, Y = np.meshgrid(x, y)
    F = np.where(ellipse_mask(X, Y, a, e), Ffn(X, Y, a, e, phi0), np.nan)
    vm = np.nanmax(np.abs(F)) or 1.0
    kw = dict(extent=[-a*1.05, a*1.05, -b*1.05, b*1.05],
              origin='lower', aspect='equal',
              cmap=cmap, interpolation='bilinear')
    ax.imshow(F, vmin=-vm if signed else 0, vmax=vm, **kw)
    _overlay(ax, a, e)
    ax.set_title(title, fontsize=8, pad=2)
    ax.set_xlabel('x (mm)', fontsize=7)
    ax.set_ylabel('y (mm)', fontsize=7)
    ax.tick_params(labelsize=6)

def draw_s11(ax, fte, ftm):
    flo = min(fte, ftm, TARGET) * 0.87
    fhi = max(fte, ftm, TARGET) * 1.13
    f   = np.linspace(flo, fhi, 3000)
    DEPTH = -35   # dB depth at resonance

    def S11_dip(fc, Q):
        d = 2 * Q * (f - fc) / fc
        return DEPTH / (1 + d**2)   # Lorentzian dip

    ax.axhline(0, color='grey', lw=0.8, alpha=0.5)
    ax.plot(f, S11_dip(fte, 9000), color='#1F77B4', lw=2, label=f'TE₁₁₁  {fte:.4f} GHz')
    ax.plot(f, S11_dip(ftm, 7000), color='#D62728', lw=2, label=f'TM₀₁₀  {ftm:.4f} GHz')
    ax.axvline(TARGET, color='#2CA02C', ls='--', lw=1.5, label=f'{TARGET} GHz')

    df = abs(fte - ftm) * 1000
    ax.annotate(f'|Δf| = {df:.3f} MHz', xy=(0.98, 0.08), xycoords='axes fraction',
                ha='right', va='bottom', fontsize=8, color='purple',
                bbox=dict(boxstyle='round,pad=0.3', fc='lightyellow', ec='purple', alpha=0.85))

    ax.set_ylim(DEPTH * 1.15, 5)
    ax.set_ylabel('S₁₁ (dB)', fontsize=8)
    ax.set_xlabel('Frequency (GHz)', fontsize=8)
    ax.set_title('S₁₁ vs Frequency  (dips downward)', fontsize=9)
    ax.legend(fontsize=7, loc='lower center')
    ax.grid(alpha=0.3)
    ax.tick_params(labelsize=7)

def draw_Lscan(ax, a, e, er, Lcur):
    LSTART = 20.0
    Lmax   = max(150.0, Lcur * 2.5)
    Ls     = np.linspace(LSTART, Lmax, 800)
    fTEs   = fTE(a, e, Ls, er)
    ftmC   = fTM(a, e, er)
    fLcur  = fTE(a, e, Lcur, er)
    cross  = Ldegen_bisect(a, e, er, Lo=LSTART, Hi=Lmax)

    key_f = [ftmC, TARGET, fLcur]
    if cross: key_f.append(cross['f'])
    fpad = 0.15
    ylo = min(1.0,  min(key_f) - fpad)
    yhi = max(5.0,  max(key_f) + fpad)

    ax.plot(Ls, fTEs, color='#1F77B4', lw=2, label='TE₁₁₁')
    ax.axhline(ftmC, color='#D62728', lw=2, label=f'TM₀₁₀ = {ftmC:.4f} GHz')
    ax.axhline(TARGET, color='#2CA02C', ls='--', lw=1.5, label=f'{TARGET} GHz target')

    if cross:
        Lc, fc = cross['L'], cross['f']
        ax.plot([LSTART, Lc], [fc, fc], color='#27AE60', lw=0.8, ls=':', alpha=0.7)
        ax.plot([Lc, Lc], [ylo, fc], color='#27AE60', lw=0.8, ls=':', alpha=0.7)
        ax.plot(Lc, fc, 'o', color='#27AE60', ms=14, zorder=8, alpha=0.3)
        ax.plot(Lc, fc, 'o', color='#27AE60', ms=9,  zorder=9, label=f'L_degen = {Lc:.2f} mm, f = {fc:.4f} GHz')
        ax.plot(Lc, fc, 'o', color='white',   ms=4,  zorder=10)
        ax.annotate(f'  ✓  L_degen = {Lc:.2f} mm\n      f = {fc:.4f} GHz',
                    xy=(Lc, fc), xytext=(min(Lc + (Lmax - LSTART) * 0.07, Lmax * 0.78), fc + (yhi - ylo) * 0.12),
                    fontsize=9, color='#145A32', fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.5', fc='#EAFAF1', ec='#27AE60', lw=1.5),
                    arrowprops=dict(arrowstyle='->', color='#27AE60', lw=1.8))
        ax.annotate(f'{Lc:.2f}', xy=(Lc, ylo), xytext=(Lc, ylo), ha='center', va='top', fontsize=8, color='#27AE60', fontweight='bold')
    else:
        ax.text(0.5, 0.5, 'No crossing in range — try larger a or smaller ε_r',
                transform=ax.transAxes, ha='center', va='center', color='#D62728', fontsize=9)

    ax.axvline(Lcur, color='orange', ls=':', lw=2, zorder=7, label=f'current L = {Lcur:.1f} mm')
    ax.plot(Lcur, fLcur, 's', color='orange',  ms=9, zorder=8)
    ax.plot(Lcur, ftmC,  's', color='#D62728', ms=9, zorder=8)

    ax.set_xlim(LSTART, Lmax)
    ax.set_ylim(ylo, yhi)
    ax.set_xlabel('Cavity height L (mm)', fontsize=9)
    ax.set_ylabel('f (GHz)',              fontsize=9)
    ax.set_title('Degeneracy scan vs L  (● green = crossing,  orange = current L slider)', fontsize=9)
    ax.legend(fontsize=7, loc='upper right')
    ax.grid(alpha=0.3)
    ax.tick_params(labelsize=8)

### Passo 6: Criação do Painel Principal e UI Interativa

Nesta seção, juntamos todos os gráficos criados no passo anterior usando o `GridSpec` para compor o Dashboard final. 

Em seguida, inicializamos os sliders dinâmicos utilizando `ipywidgets` (caso esteja num ambiente interativo como Jupyter) para manipular parâmetros como Eixo Semimaior ($a$), excentricidade ($e$), permissividade ($\varepsilon_r$), etc., em tempo real.

In [ ]:
# ── master figure ─────────────────────────────────────────────
def build_figure(a, e, L, er, phi0, N=160):
    plt.close('all')

    fte  = fTE(a, e, L, er)
    ftm  = fTM(a, e, er)
    df   = abs(fte - ftm) * 1000
    vm   = mode_volume(a, e, L)
    onT  = abs(ftm - TARGET) / TARGET < 0.015
    degen = df < 20

    if degen and onT:
        st, sc = f'✓ DEGENERATE @ 2.87 GHz   |Δf| = {df:.3f} MHz', 'green'
    elif degen:
        st, sc = f'~ Degenerate mas fora do alvo   f(TM) = {ftm:.4f} GHz', 'orange'
    else:
        st, sc = f'✗ Not degenerate   |Δf| = {df:.3f} MHz', 'red'

    b = a * np.sqrt(1 - e**2)
    fig = plt.figure(figsize=(18, 11))
    fig.suptitle(
        f'a={a:.2f} mm   b={b:.2f} mm   L={L:.2f} mm   '
        f'e={e:.3f}   ε_r={er:.2f}   '
        f'f(TE)={fte:.4f} GHz   f(TM)={ftm:.4f} GHz\n{st}',
        fontsize=10, fontweight='bold', color=sc)

    gs = GridSpec(3, 3, figure=fig, hspace=0.55, wspace=0.35)

    # TE11 fields
    draw_field(fig.add_subplot(gs[0, 0]), F_Eperp, a, e, phi0, '|E_⊥| TE₁₁ — two-lobe transverse E', N=N)
    draw_field(fig.add_subplot(gs[0, 1]), F_Hz, a, e, phi0, 'H_z TE₁₁ — axial H (signed)', cmap='RdBu_r', signed=True, N=N)
    draw_field(fig.add_subplot(gs[0, 2]), F_Hperp, a, e, phi0, '|H_⊥| TE₁₁ — NV spin coupling ∝ ∫|H_⊥|²dV', N=N)

    # TM01 fields + S11
    draw_field(fig.add_subplot(gs[1, 0]), F_Ez, a, e, phi0, 'E_z TM₀₁ — dome (zero at wall)', N=N)
    draw_field(fig.add_subplot(gs[1, 1]), F_Hphi, a, e, phi0, '|H_φ| TM₀₁ — annular ring', N=N)
    draw_s11(fig.add_subplot(gs[1, 2]), fte, ftm)

    # L-scan full bottom row
    draw_Lscan(fig.add_subplot(gs[2, :]), a, e, er, L)

    return fig

# ── interface and execution ────────────────────────────────────
def run_interactive():
    style = {'description_width': '190px'}
    lay   = widgets.Layout(width='560px')

    sl = dict(
        er  = widgets.FloatSlider(value=9.4, min=1.0, max=90.0, step=0.05, description='ε_r  (1=air, 9.4=sapphire):', style=style, layout=lay),
        a   = widgets.FloatSlider(value=14.2, min=1.0, max=60.0, step=0.02, description='a — semi-major axis (mm):', style=style, layout=lay),
        e   = widgets.FloatSlider(value=0.54, min=0.01, max=0.96, step=0.001, description='e — eccentricity:', style=style, layout=lay),
        L   = widgets.FloatSlider(value=26.4, min=1.0, max=150.0, step=0.05, description='L — height (mm):', style=style, layout=lay),
        phi = widgets.FloatSlider(value=0.0, min=0.0, max=180.0, step=1.0, description='φ₀ — TE polarisation (°):', style=style, layout=lay),
    )
    out = widgets.Output()

    def update_plot(change=None):
        with out:
            clear_output(wait=True)
            fig = build_figure(sl['a'].value, sl['e'].value, sl['L'].value, sl['er'].value, np.radians(sl['phi'].value))
            display(fig)
            plt.close(fig)

    for s in sl.values():
        s.observe(update_plot, names='value')

    display(widgets.VBox([
        widgets.HTML('<h3>Elliptical Cavity NV-Maser Dashboard</h3>'),
        *sl.values(),
        out
    ]))
    update_plot()

# Ponto de entrada do script
if __name__ == '__main__':
        run_interactive()
